In [6]:
#importing necessary libraries
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import os

# importing functions
from configuration import readfile2, retrievedata2, thresholdtrim, timearray
from normalization import lifetime2
from plotting import plt, comparison_plt
from fitting import lifetime_fit
from usr_interactive import confirm_term, lifetime_def_params, terminate_program

In [7]:
""""
Main Program:
This program is designed to deal with experimental data collected in T2 mode for lifetime measurements.
"""

if __name__ == '__main__':
    # termination command - the program will check to see if the user terminated the program whenever it asks for a user input
    print("To terminate this program press '/")
    # terminate variable represented as a boolean
    terminate = False 

    # identifies the number corresponding to the lifetime measurement - this value will be needed for certain functions
    measurement_type = 2

    while terminate == False:
    # - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - #
    # - - - - - - - - - - - - - - - - - - - - - - - - - - file information given by user - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - #
    # - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - #
        # - - - - - - - background data information - - - - - - - #
        # background measurement information #
        ##### background file path #####

        # defining the prompt for confirmation function
        bckgrd_file_prompt = 'the file path for the background data:'
        # asking for the background file path
        background_filepath = confirmation(bckgrd_file_prompt)
        #background_filepath = ''

        # check for termination input
        terminate = terminate_program(background_filepath)
        if terminate == True:
            break

        ##### background file duration #####

        # defining the prompt for confirmation function
        bckgrd_duration_prompt = 'the duration of the background measurement as a number in minutes:'
        # asking for the background experiment duration
        background_exp_duration = confirmation(bckgrd_duration_prompt)

        # check for termination input
        terminate = terminate_program(background_exp_duration)
        if terminate == True:
            break

        # conversion to float to allow for calculations
        background_exp_duration = float(background_exp_duration)

        # converting to seconds
        background_exp_duration = background_exp_duration * 60

        # - - - - - - - experimental data information - - - - - - - #
        # experimental measurement information
        ##### experimental file path #####

        # defining the prompt for confirmation function
        exp_file_prompt = 'the file path for the experimental data:'
        # asking for the experimental file path
        experimental_filepath = confirmation(exp_file_prompt)

        terminate = terminate_program(experimental_filepath)
        if terminate == True:
            break

        ##### experiemental file duration #####
        
        # defining the prompt for confirmation function
        experimental_duration_prompt = 'the duration of the experiment measurement as a number in minutes:'
        # asking for the background experiment duration
        experimental_exp_duration = confirmation(experimental_duration_prompt)

        # check for termination input
        terminate = terminate_program(experimental_exp_duration)
        if terminate == True:
            break

        # converstion to float to allow for calculations
        experimental_exp_duration = float(experimental_exp_duration)

        # converstion to seconds
        experimental_exp_duration = experimental_exp_duration * 60

        # after correct conversions of duration times
        duration_ratio = experimental_exp_duration / background_exp_duration

        # retrieving data and experimental information from files
        print("")
        # using reading function to retrieve measurement information - returns a list in the form [date, time, channels_per_curve, ns_channel]
        background_exp_info = reading_file(background_filepath)

        # printing background measurement information
        print("Background Measurement Information:")
        print("    Date and Time of Measurement:", background_exp_info[0], background_exp_info[1])
        print("    channels_per_curve:", background_exp_info[2])
        print("    ns_channel:", background_exp_info[3])

        experimental_exp_info = reading_file(experimental_filepath)

        # printing experimental measurement information
        print("Experimental Measurement Information:")
        print("     Date and Time of Measurement:", experimental_exp_info[0], experimental_exp_info[1])
        print("     channels_per_curve:", experimental_exp_info[2])
        print("     ns_channel:", experimental_exp_info[3])

        print("")

    # - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - #
    # - - - - - - - - - - - - - - - - - - - - - - - - - - Reading in Data & Normalization of Data - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - #
    # - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - #

        # reading in experimental and background data
        background_data = retrieve_data(background_filepath)
        experimental_data = retrieve_data(experimental_filepath)

        # normalizing the data
        normalized_lifetime_data = T2_lifetime_normalized(experimental_data, background_data, duration_ratio)

        # trimming some of the data after it has plateaued for a specific interval
        trimmed_normalized_lifetime_data = threshold_trim_function(normalized_lifetime_data)

        # defining the time array
        lifetime_time_array = time_array_construction(1, len(trimmed_normalized_lifetime_data), experimental_exp_info[3], 0)

        # prints out the set guesses if the user does not wish to define their own

        print("Set Guesses:")
        print("")

        print("Exponential Fit:")
        print("A1 = 500, τ1 = 100")

        print("")

        print("Bi-exponential Fit:")
        print("A1 = 500, τ1 = 100, A2 = 50, τ2 = 10")

        print("")

        print("Tri-exponential Fit:")
        print("A1 = 500, τ1 = 100, A2 = 50, τ2 = 10, A3 = 20, τ3 = 5")

        print("")

        print("Please input either Y or N.")

        # prompts the user to tell the program whether or not they wish to define their own fits
        # if not the program uses a while loop s.t. it won't crash if the user does not give a correct input

        valid = False
        while valid == False:
            path = input("Would you like to use the set guesses for your exponential fits?")
            path = path.upper()
            if path == 'Y':
                valid = True
            elif path == 'N':
                valid = True
            else:
                print("You did not enter a valid input.")

        #if the user wishes to use the set guess the program will follow this path
        if path == 'Y':
            # in the form A1, tau1, A2, tau2, A3, tau3
            lifetime_set_guesses_exp = [500, 100, 0, 0, 0, 0]
            lifetime_set_guesses_bi = [500, 100, 50, 10, 0, 0]
            lifetime_set_guesses_tri = [500, 100, 50, 10, 20, 5]

            # defining the fit models in the form: 
            # [number of parameters, 2D array of parameter values and their errors, the normalized fit, residuals]
            set_fit1_lifetime_model = T2_lifetime_data_analysis(1, trimmed_normalized_lifetime_data, lifetime_time_array, lifetime_set_guesses_exp)
            set_fit2_lifetime_model = T2_lifetime_data_analysis(2, trimmed_normalized_lifetime_data, lifetime_time_array, lifetime_set_guesses_bi)
            set_fit3_lifetime_model = T2_lifetime_data_analysis(3, trimmed_normalized_lifetime_data, lifetime_time_array, lifetime_set_guesses_tri)

            # printing the time constants for each of the fits

            print("")

            plots(1, set_fit1_lifetime_model[0], lifetime_time_array, trimmed_normalized_lifetime_data, set_fit1_lifetime_model[2], set_fit1_lifetime_model[3])

            print("Exponential Fit Time Decay Constant:")
            print(f"τ1 = {set_fit1_lifetime_model[1][1]} ± {set_fit1_lifetime_model[1][2][1]:.2f} ns")

            print("")

            plots(1, set_fit2_lifetime_model[0], lifetime_time_array, trimmed_normalized_lifetime_data, set_fit2_lifetime_model[2], set_fit2_lifetime_model[3])

            print("Bi-Exponential Fit Time Decay Constants:")
            print(f"τ1 = {set_fit2_lifetime_model[1][1]} ± {set_fit2_lifetime_model[1][4][1]:.2f} ns")
            print(f"τ2 = {set_fit2_lifetime_model[1][3]} ± {set_fit2_lifetime_model[1][4][3]:.2f} ns")

            print("")

            plots(1, set_fit3_lifetime_model[0], lifetime_time_array, trimmed_normalized_lifetime_data, set_fit3_lifetime_model[2], set_fit3_lifetime_model[3])

            print("Tri-Exponential Fit Decay Constants:")
            print(f"τ1 = {set_fit3_lifetime_model[1][1]} ± {set_fit3_lifetime_model[1][6][1]:.2f} ns")
            print(f"τ2 = {set_fit3_lifetime_model[1][3]} ± {set_fit3_lifetime_model[1][6][3]:.2f} ns")
            print(f"τ3 = {set_fit3_lifetime_model[1][5]} ± {set_fit3_lifetime_model[1][6][5]:.2f} ns")

            print("")

            # makes the comparison of the fits plot
            comparison_plot(1, 3, lifetime_time_array, trimmed_normalized_lifetime_data, 1, set_fit1_lifetime_model[2], 2, set_fit2_lifetime_model[2], 3, set_fit3_lifetime_model[2])

        # if the user wishes to define their own guesses the program will follow this path
        elif path == 'N':
            # determines how many fits the user would like to plot (works for 1-3 fits)
            valid_input = False
            while valid_input == False:
                num_of_fits = input("How many exponential fits would you like to plot?")

                # checks for termination input
                terminate = terminate_program(num_of_fits)
                if terminate == True:
                    break

                # checks to make sure the user inputted the right input
                print("Please enter 'y' or 'n'.")
                print("You entered", num_of_fits,".")
                confirmation = input("Is this the correct number of fits you would like to plot?")

                # checks for termination input
                terminate = terminate_program(num_of_fits)
                if terminate == True:
                    break

                try:
                    num_of_fits = int(num_of_fits)
                    break
                except ValueError:
                    print("Invalid input. Please enter an integer.")

            if terminate == True:
                break

            num_of_fits = int(num_of_fits)

            fits_parameters = []

            # determines the parameters for each fit
            for n in range(num_of_fits):
                term = n + 1
                print("For your n =", term,"exponential fit:")
                parameters = T2_lifetime_def_parameters()
                if parameters == True:
                    break
                fits_parameters.append(parameters)

            fitted_models = []

            # gets the fitted models for each of the fits determined by the user
            for n in range(num_of_fits):
                fit_model = T2_lifetime_data_analysis(fits_parameters[n][0], trimmed_normalized_lifetime_data, lifetime_time_array, fits_parameters[n][1])
                plots(1, fit_model[0], lifetime_time_array, trimmed_normalized_lifetime_data, fit_model[2], fit_model[3])
                fitted_models.append(fit_model)

            # printing the time constants for the respective number of fits that have been made
            # if there is more than one exponential fit the program will also plot a comparison plot of the fits

            # only one exponential fit
            if num_of_fits == 1:
                fit1 = fitted_models[0]
                fit1_data = fit1[2]

                if fit1[0] == 1:
                    print("Exponential Fit Decay Constants:")
                    print(f"τ1 = {fit1[1][1]} ± {fit1[1][2][1]:.2f} ns")
                elif fit1[0] == 2:
                    print("Biexponential Fit Decay Constants")
                    print(f"τ1 = {fit1[1][1]} ± {fit1[1][4][1]:.2f} ns")
                    print(f"τ2 = {fit1[1][3]} ± {fit1[1][4][3]:.2f} ns")
                else:
                    print("Triexponential Fit Decay Constants")
                    print(f"τ1 = {fit1[1][1]} ± {fit1[1][6][1]:.2f} ns")
                    print(f"τ2 = {fit1[1][4]} ± {fit1[1][6][3]:.2f} ns")
                    print(f"τ3 = {fit1[1][5]} ± {fit1[1][6][5]:.2f} ns")

            # two exponential fits
            elif num_of_fits == 2:
                fit1 = fitted_models[0]
                fit1_data = fit1[2]
                fit1_num_of_exp = fit1[0]

                if fit1[0] == 1:
                    print("Exponential Fit Decay Constants:")
                    print(f"τ1 = {fit1[1][1]} ± {fit1[1][2][1]:.2f} ns")
                elif fit1[0] == 2:
                    print("Biexponential Fit Decay Constants")
                    print(f"τ1 = {fit1[1][1]} ± {fit1[1][4][1]:.2f} ns")
                    print(f"τ2 = {fit1[1][3]} ± {fit1[1][4][3]:.2f} ns")
                else:
                    print("Triexponential Fit Decay Constants")
                    print(f"τ1 = {fit1[1][1]} ± {fit1[1][6][1]:.2f} ns")
                    print(f"τ2 = {fit1[1][4]} ± {fit1[1][6][3]:.2f} ns")
                    print(f"τ3 = {fit1[1][5]} ± {fit1[1][6][5]:.2f} ns")

                fit2 = fitted_models[1]
                fit2_data = fit2[2]
                fit2_num_of_exp = fit2[0]

                if fit2[0] == 1:
                    print("Exponential Fit Decay Constants:")
                    print(f"τ1 = {fit2[1][1]} ± {fit2[1][2][1]:.2f} ns")
                elif fit2[0] == 2:
                    print("Biexponential Fit Decay Constants")
                    print(f"τ1 = {fit2[1][1]} ± {fit2[1][4][1]:.2f} ns")
                    print(f"τ2 = {fit2[1][3]} ± {fit2[1][4][3]:.2f} ns")
                else:
                    print("Triexponential Fit Decay Constants")
                    print(f"τ1 = {fit2[1][1]} ± {fit2[1][6][1]:.2f} ns")
                    print(f"τ2 = {fit2[1][4]} ± {fit2[1][6][3]:.2f} ns")
                    print(f"τ3 = {fit2[1][5]} ± {fit2[1][6][5]:.2f} ns")

                # plotting a comparison plot
                comparison_plot(1, num_of_fits, lifetime_time_array, trimmed_normalized_lifetime_data, fit1_num_of_exp, fit1_data, fit2_num_of_exp, fit2_data, 0, 0)

            # three exponential fits
            elif num_of_fits == 3:
                fit1 = fitted_models[0]
                fit1_data = fit1[2]
                fit1_num_of_exp = fit1[0]

                if fit1[0] == 1:
                    print("Exponential Fit Decay Constants:")
                    print(f"τ1 = {fit1[1][1]} ± {fit1[1][2][1]:.2f} ns")
                elif fit1[0] == 2:
                    print("Biexponential Fit Decay Constants")
                    print(f"τ1 = {fit1[1][1]} ± {fit1[1][4][1]:.2f} ns")
                    print(f"τ2 = {fit1[1][3]} ± {fit1[1][4][3]:.2f} ns")
                else:
                    print("Triexponential Fit Decay Constants")
                    print(f"τ1 = {fit1[1][1]} ± {fit1[1][6][1]:.2f} ns")
                    print(f"τ2 = {fit1[1][4]} ± {fit1[1][6][3]:.2f} ns")
                    print(f"τ3 = {fit1[1][5]} ± {fit1[1][6][5]:.2f} ns")

                fit2 = fitted_models[1]
                fit2_data = fit2[2]
                fit2_num_of_exp = fit2[0]

                if fit2[0] == 1:
                    print("Exponential Fit Decay Constants:")
                    print(f"τ1 = {fit2[1][1]} ± {fit2[1][2][1]:.2f} ns")
                elif fit2[0] == 2:
                    print("Biexponential Fit Decay Constants")
                    print(f"τ1 = {fit2[1][1]} ± {fit2[1][4][1]:.2f} ns")
                    print(f"τ2 = {fit2[1][3]} ± {fit2[1][4][3]:.2f} ns")
                else:
                    print("Triexponential Fit Decay Constants")
                    print(f"τ1 = {fit2[1][1]} ± {fit2[1][6][1]:.2f} ns")
                    print(f"τ2 = {fit2[1][4]} ± {fit2[1][6][3]:.2f} ns")
                    print(f"τ3 = {fit2[1][5]} ± {fit2[1][6][5]:.2f} ns")

                fit3 = fitted_models[2]
                fit3_data = fit3[2]
                fit3_num_of_exp = fit3[0]

                if fit3[0] == 1:
                    print("Exponential Fit Decay Constants:")
                    print(f"τ1 = {fit3[1][1]} ± {fit3[1][2][1]:.2f} ns")
                elif fit3[0] == 2:
                    print("Biexponential Fit Decay Constants")
                    print(f"τ1 = {fit3[1][1]} ± {fit3[1][4][1]:.2f} ns")
                    print(f"τ2 = {fit3[1][3]} ± {fit3[1][4][3]:.2f} ns")
                else:
                    print("Triexponential Fit Decay Constants")
                    print(f"τ1 = {fit3[1][1]} ± {fit3[1][6][1]:.2f} ns")
                    print(f"τ2 = {fit3[1][4]} ± {fit3[1][6][3]:.2f} ns")
                    print(f"τ3 = {fit3[1][5]} ± {fit3[1][6][5]:.2f} ns")
            
                # plotting a comparison plot
                comparison_plot(1, num_of_fits, lifetime_time_array, trimmed_normalized_lifetime_data, fit1[0], fit1_data, fit2[0], fit2_data, fit3[0], fit3_data)

        # sets terminate to True at the end of the program s.t. the program terminates after all data analysis and plotting is complete
        terminate = True

To terminate this program press '/


NameError: name 'confirmation' is not defined